# Cloud Vision AI 핸즈온: GTIN/UPC 추출 및 상품 정보 기록

Google Cloud Vision API를 활용하여 상품 이미지에서 바코드(GTIN, UPC, EAN 등)를 인식하고,
상품명·상세 정보를 자동으로 추출하는 핸즈온 노트북입니다.

## Gemini OCR과의 차이점

| 항목 | Gemini OCR | Cloud Vision AI |
|------|-----------|-----------------|
| 방식 | 생성형 AI (LLM) | 전통적 CV + ML 파이프라인 |
| OCR | 프롬프트 기반 자유 형식 | `TEXT_DETECTION` / `DOCUMENT_TEXT_DETECTION` |
| 객체 감지 | 프롬프트로 bounding box 요청 | `OBJECT_LOCALIZATION`, `LABEL_DETECTION` |
| 로고/브랜드 | 프롬프트로 추론 | `LOGO_DETECTION` 전용 기능 |
| 바코드 값 | LLM이 직접 해석 | OCR 텍스트에서 정규식으로 파싱 |
| 비용 | 토큰 기반 과금 | 요청 건수 기반 과금 (월 1,000건 무료) |

## 목차
1. Setup
2. 공통 함수 정의
3. Label / Logo Detection — 상품 인식
4. OCR — GTIN/UPC 추출 및 상품 정보 기록

---
## 1. Setup

필요한 라이브러리를 설치하고 Cloud Vision API 클라이언트를 초기화합니다.

| 패키지 | 역할 |
|--------|------|
| `google-cloud-vision` | Google Cloud Vision API 클라이언트. 이미지에서 텍스트, 라벨, 로고, 객체 등을 감지합니다. |
| `opencv-python` | 이미지 읽기, 색상 변환, bounding box 그리기 등 컴퓨터 비전 처리에 사용합니다. |
| `Pillow` | Python 이미지 처리 라이브러리(PIL). 이미지를 열고, 변환하고, 노트북에서 출력하는 데 사용합니다. |

In [ ]:
!pip install -q google-cloud-vision opencv-python Pillow

In [ ]:
import json
import re
from pathlib import Path

import cv2
import numpy as np
import requests
from google.cloud import vision
from google.oauth2 import service_account
from PIL import Image

In [ ]:
# Cloud Vision API 클라이언트 초기화
# 방법 1: 서비스 계정 키 파일 경로 지정
# SERVICE_ACCOUNT_KEY = "path/to/your-service-account-key.json"  # TODO: 본인의 키 파일 경로
# credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_KEY)
# client = vision.ImageAnnotatorClient(credentials=credentials)

# 방법 2: GOOGLE_APPLICATION_CREDENTIALS 환경변수가 설정된 경우 (Colab/Vertex AI Workbench 등)
client = vision.ImageAnnotatorClient()

---
## 2. 공통 함수 정의

In [ ]:
BOX_COLORS = [
    (255, 56, 56), (255, 157, 151), (255, 112, 31), (255, 178, 29),
    (207, 210, 49), (72, 249, 10), (146, 204, 23), (61, 219, 134),
    (26, 147, 52), (0, 212, 187), (44, 153, 168), (0, 194, 255),
    (52, 69, 147), (100, 115, 255), (0, 24, 236), (132, 56, 255),
    (82, 0, 133), (203, 56, 255), (255, 149, 200), (255, 55, 199),
]

GTIN_PATTERNS = {
    "GTIN-14": re.compile(r"\b\d{14}\b"),
    "GTIN-13 / EAN-13": re.compile(r"\b\d{13}\b"),
    "GTIN-12 / UPC-A": re.compile(r"\b\d{12}\b"),
    "GTIN-8 / EAN-8": re.compile(r"\b\d{8}\b"),
}


def load_vision_image(filename):
    """파일을 Cloud Vision API용 Image 객체로 로드합니다."""
    path = Path(filename)
    content = path.read_bytes()
    return vision.Image(content=content)


def download_image(url, filename):
    """URL에서 이미지를 다운로드합니다."""
    path = Path(filename)
    if not path.exists():
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        path.write_bytes(resp.content)
    return filename


def read_image(filename):
    """이미지를 읽어 PIL Image와 너비/높이를 반환합니다."""
    image = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    return Image.fromarray(image), w, h


def draw_poly_boxes(pil_image, annotations, color_idx_start=0):
    """Vision API의 bounding_poly 결과를 이미지 위에 그립니다."""
    img = np.array(pil_image)
    for idx, ann in enumerate(annotations):
        verts = ann.bounding_poly.vertices
        pts = np.array([(v.x, v.y) for v in verts], dtype=np.int32)
        color = BOX_COLORS[(color_idx_start + idx) % len(BOX_COLORS)]
        cv2.polylines(img, [pts], True, color, 2)

        label = ann.description if hasattr(ann, "description") else str(ann.score)
        x, y = pts[0]
        (tw, th), _ = cv2.getTextSize(label[:40], cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x, y - th - 6), (x + tw, y), color, -1)
        cv2.putText(img, label[:40], (x, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return Image.fromarray(img)


def draw_normalized_boxes(pil_image, objects, w, h):
    """Vision API OBJECT_LOCALIZATION의 normalized bounding_poly를 그립니다."""
    img = np.array(pil_image)
    for idx, obj in enumerate(objects):
        verts = obj.bounding_poly.normalized_vertices
        pts = np.array([(int(v.x * w), int(v.y * h)) for v in verts], dtype=np.int32)
        color = BOX_COLORS[idx % len(BOX_COLORS)]
        cv2.polylines(img, [pts], True, color, 2)

        label = f"{obj.name} ({obj.score:.0%})"
        x, y = pts[0]
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img, (x, y - th - 6), (x + tw, y), color, -1)
        cv2.putText(img, label, (x, y - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    return Image.fromarray(img)


def extract_gtin_codes(text):
    """텍스트에서 GTIN/UPC/EAN 패턴의 숫자 코드를 추출합니다."""
    found = []
    for code_type, pattern in GTIN_PATTERNS.items():
        matches = pattern.findall(text)
        for m in matches:
            found.append({"type": code_type, "value": m})
    return found

---
## 3. Label / Logo Detection — 상품 인식

Cloud Vision API의 `LABEL_DETECTION`, `LOGO_DETECTION`, `OBJECT_LOCALIZATION` 기능을 사용하여
이미지에서 상품 카테고리, 브랜드, 객체를 감지합니다.

| Feature | 설명 |
|---------|------|
| `LABEL_DETECTION` | 이미지 내 사물/장면/개념을 라벨로 분류 |
| `LOGO_DETECTION` | 브랜드 로고를 감지하고 위치를 반환 |
| `OBJECT_LOCALIZATION` | 개별 객체를 감지하고 bounding box를 반환 |

In [ ]:
# 테스트 이미지 다운로드 (원하는 상품 이미지로 교체 가능)
download_image(
    "https://github.com/ultralytics/notebooks/releases/download/v0.0.0/gemini-image4.jpg",
    "gemini-image4.jpg",
)

caption_image, cap_w, cap_h = read_image("gemini-image4.jpg")
display(caption_image)

### 3-1. Label Detection — 이미지 라벨 분류

In [ ]:
vision_image = load_vision_image("gemini-image4.jpg")

label_response = client.label_detection(image=vision_image)

print("=== Label Detection 결과 ===\n")
for label in label_response.label_annotations:
    print(f"  {label.description:<30s}  신뢰도: {label.score:.1%}")

### 3-2. Logo Detection — 브랜드 로고 감지

In [ ]:
logo_response = client.logo_detection(image=vision_image)

logos = logo_response.logo_annotations
if logos:
    print("=== Logo Detection 결과 ===\n")
    for logo in logos:
        print(f"  브랜드: {logo.description}  신뢰도: {logo.score:.1%}")
    display(draw_poly_boxes(caption_image, logos))
else:
    print("로고가 감지되지 않았습니다.")

### 3-3. Object Localization — 객체 감지 및 위치 표시

In [ ]:
object_response = client.object_localization(image=vision_image)

objects = object_response.localized_object_annotations
if objects:
    print("=== Object Localization 결과 ===\n")
    for obj in objects:
        print(f"  {obj.name:<25s}  신뢰도: {obj.score:.1%}")
    display(draw_normalized_boxes(caption_image, objects, cap_w, cap_h))
else:
    print("객체가 감지되지 않았습니다.")

---
## 4. OCR — GTIN/UPC 추출 및 상품 정보 기록

Cloud Vision API의 `TEXT_DETECTION` 기능을 사용하여 상품 이미지에서 텍스트를 추출하고,
정규식으로 GTIN/UPC/EAN 코드를 파싱합니다.

`TEXT_DETECTION`은 이미지 내 모든 텍스트를 감지하며, 각 단어/문장의 bounding box도 반환합니다.

In [ ]:
# 바코드 테스트 이미지 다운로드 (원하는 상품 바코드 이미지로 교체 가능)
download_image(
    "https://free-barcode.com/barcode/barcode-technology/detail-gtin-14-barcode/5.jpg",
    "barcode_image.jpg",
)

ocr_image, ocr_w, ocr_h = read_image("barcode_image.jpg")
display(ocr_image)

### 4-1. TEXT_DETECTION — 전체 텍스트 추출 및 시각화

이미지에서 감지된 모든 텍스트를 추출하고, 각 단어의 위치를 bounding box로 표시합니다.

In [ ]:
barcode_vision_image = load_vision_image("barcode_image.jpg")

text_response = client.text_detection(image=barcode_vision_image)
texts = text_response.text_annotations

if texts:
    full_text = texts[0].description
    print("=== 전체 OCR 텍스트 ===\n")
    print(full_text)
    print("\n=== 개별 텍스트 영역 (bounding box 포함) ===\n")
    word_annotations = texts[1:]
    for t in word_annotations:
        verts = t.bounding_poly.vertices
        coords = [(v.x, v.y) for v in verts]
        print(f"  \"{t.description}\"  위치: {coords}")

    display(draw_poly_boxes(ocr_image, word_annotations))
else:
    print("텍스트가 감지되지 않았습니다.")

### 4-2. GTIN/UPC/EAN 코드 자동 추출

OCR로 추출한 텍스트에서 정규식 패턴 매칭을 통해 GTIN/UPC/EAN 코드를 자동으로 식별합니다.

| 코드 유형 | 자릿수 | 설명 |
|-----------|--------|------|
| GTIN-14 | 14자리 | 물류 단위 (박스/팔레트) |
| GTIN-13 / EAN-13 | 13자리 | 국제 표준 상품 코드 |
| GTIN-12 / UPC-A | 12자리 | 북미 표준 상품 코드 |
| GTIN-8 / EAN-8 | 8자리 | 소형 상품용 축약 코드 |

In [ ]:
if texts:
    gtin_codes = extract_gtin_codes(full_text)

    if gtin_codes:
        print("=== 추출된 GTIN/UPC/EAN 코드 ===\n")
        for code in gtin_codes:
            print(f"  유형: {code['type']:<20s}  값: {code['value']}")
    else:
        print("GTIN/UPC/EAN 패턴의 코드가 발견되지 않았습니다.")
        print("OCR 텍스트에서 숫자만 추출합니다:\n")
        numbers = re.findall(r"\d+", full_text)
        for n in numbers:
            print(f"  숫자: {n}")
else:
    print("텍스트가 감지되지 않았습니다.")

### 4-3. 상품 정보 종합 — 구조화된 JSON 기록

Label Detection, Logo Detection, OCR 결과를 결합하여 상품 정보를 구조화된 JSON으로 기록합니다.

In [ ]:
def analyze_product(image_path):
    """Cloud Vision API의 여러 기능을 조합하여 상품 정보를 종합 분석합니다."""
    vision_img = load_vision_image(image_path)

    # 여러 Feature를 한 번의 API 호출로 요청 (배치 처리)
    features = [
        vision.Feature(type_=vision.Feature.Type.TEXT_DETECTION),
        vision.Feature(type_=vision.Feature.Type.LABEL_DETECTION, max_results=10),
        vision.Feature(type_=vision.Feature.Type.LOGO_DETECTION, max_results=5),
        vision.Feature(type_=vision.Feature.Type.OBJECT_LOCALIZATION, max_results=10),
    ]
    request = vision.AnnotateImageRequest(image=vision_img, features=features)
    response = client.annotate_image(request=request)

    # OCR 텍스트 추출
    ocr_text = ""
    if response.text_annotations:
        ocr_text = response.text_annotations[0].description

    # GTIN/UPC 코드 추출
    gtin_codes = extract_gtin_codes(ocr_text)

    # 라벨 추출
    labels = [
        {"label": l.description, "confidence": round(l.score, 3)}
        for l in response.label_annotations
    ]

    # 로고 추출
    logos = [
        {"brand": l.description, "confidence": round(l.score, 3)}
        for l in response.logo_annotations
    ]

    # 객체 추출
    objects = [
        {"name": o.name, "confidence": round(o.score, 3)}
        for o in response.localized_object_annotations
    ]

    product_info = {
        "image_path": image_path,
        "gtin_upc_codes": gtin_codes,
        "ocr_full_text": ocr_text.strip(),
        "labels": labels,
        "logos": logos,
        "objects": objects,
    }

    return product_info


product_info = analyze_product("barcode_image.jpg")
print(json.dumps(product_info, indent=2, ensure_ascii=False))

### 4-4. 여러 이미지 일괄 처리 (Batch)

Cloud Vision API의 `batch_annotate_images`를 사용하면 여러 이미지를 한 번의 API 호출로 처리할 수 있습니다.

In [ ]:
def batch_analyze_products(image_paths):
    """여러 이미지를 한 번의 API 호출로 일괄 분석합니다."""
    features = [
        vision.Feature(type_=vision.Feature.Type.TEXT_DETECTION),
        vision.Feature(type_=vision.Feature.Type.LABEL_DETECTION, max_results=10),
        vision.Feature(type_=vision.Feature.Type.LOGO_DETECTION, max_results=5),
    ]

    requests = []
    for path in image_paths:
        vision_img = load_vision_image(path)
        req = vision.AnnotateImageRequest(image=vision_img, features=features)
        requests.append(req)

    batch_response = client.batch_annotate_images(requests=requests)

    results = []
    for path, resp in zip(image_paths, batch_response.responses):
        ocr_text = ""
        if resp.text_annotations:
            ocr_text = resp.text_annotations[0].description

        gtin_codes = extract_gtin_codes(ocr_text)
        labels = [l.description for l in resp.label_annotations[:5]]
        logos = [l.description for l in resp.logo_annotations]

        results.append({
            "image_path": path,
            "gtin_upc_codes": gtin_codes,
            "ocr_text_preview": ocr_text.strip()[:200],
            "top_labels": labels,
            "logos": logos,
        })

    return results


# 예시: 다운로드한 이미지들로 배치 처리
batch_results = batch_analyze_products(["barcode_image.jpg", "gemini-image4.jpg"])

for result in batch_results:
    print(f"\n{'='*60}")
    print(json.dumps(result, indent=2, ensure_ascii=False))

### 4-5. 나만의 상품 이미지로 테스트

아래 셀에서 `image_path`를 변경하여 본인의 상품 이미지를 테스트해 보세요.

In [ ]:
# image_path = "your_product_image.jpg"  # TODO: 본인의 이미지 경로로 변경
# my_image, w, h = read_image(image_path)
# display(my_image)
#
# # 종합 분석
# print("=== 상품 종합 분석 ===")
# product_info = analyze_product(image_path)
# print(json.dumps(product_info, indent=2, ensure_ascii=False))
#
# # OCR 텍스트 영역 시각화
# vision_img = load_vision_image(image_path)
# text_resp = client.text_detection(image=vision_img)
# if text_resp.text_annotations:
#     display(draw_poly_boxes(my_image, text_resp.text_annotations[1:]))